In [1]:
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import (
train_test_split,
 cross_val_score,
 GridSearchCV
 )
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import(
    accuracy_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

| Parameter                 | One-line Answer                                           |
| ------------------------- | --------------------------------------------------------- |
| `n_estimators=100`        | Builds 100 decision trees.                                |
| `criterion="gini"`        | Uses Gini Impurity to choose the best split.              |
| `max_depth=8`             | Limits each tree to 8 levels.                             |
| `min_samples_split=5`     | A node needs at least 5 samples to split.                 |
| `min_samples_leaf=2`      | Each leaf must have at least 2 samples.                   |
| `max_features="sqrt"`     | Uses the square root of the total features at each split. |
| `bootstrap=True`          | Trains each tree on a random sample with replacement.     |
| `oob_score=True`          | Uses unused samples to estimate model accuracy.           |
| `random_state=42`         | Makes results reproducible.                               |
| `class_weight="balanced"` | Gives more weight to minority classes.                    |


In [9]:
df=pd.read_csv("diabetes_cleaned.csv")
print(df.head())


X=df.drop("Outcome", axis=1)
y=df["Outcome"]


X_train,X_test,y_train,y_test=train_test_split(
    X,y,train_size=0.20,
random_state=42,
stratify=y

)

model=RandomForestClassifier(
    n_estimators=100,
    criterion="gini",
    max_depth=8,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features="sqrt",
    bootstrap=True,
    oob_score=True,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train,y_train)
y_pred=model.predict(X_test)
y_probability=model.predict_proba(X_test)


print("Accuracy:", accuracy_score(y_test,y_pred))

print("precision_score",precision_score(y_test,y_pred))
print("confusion_matrix",confusion_matrix(y_test,y_pred))
print("classification_report",classification_report(y_test,y_pred))
print("recall_score",recall_score(y_test,y_pred))
print("f1_score",f1_score(y_test,y_pred))


   Pregnancies  Glucose  BloodPressure  SkinThickness     Insulin   BMI  \
0            6    148.0           72.0       35.00000  155.548223  33.6   
1            1     85.0           66.0       29.00000  155.548223  26.6   
2            8    183.0           64.0       29.15342  155.548223  23.3   
3            1     89.0           66.0       23.00000   94.000000  28.1   
4            0    137.0           40.0       35.00000  168.000000  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
Accuracy: 0.7414634146341463
precision_score 0.6196581196581197
confusion_matrix [[311  89]
 [ 70 145]]
classification_report               precision    recall  f1-score   support

           0       0.82      0.78      0.80       400
           1       0.62      0.67      0.65       

In [11]:
roc = roc_auc_score(y_test, y_probability[:,1])
print("ROC SCORE",roc)
scores=cross_val_score(

    model,X,y,cv=5,scoring='accuracy'
)
print("\nCross Validation Scores")

print(scores)

print("\nAverage Cross Validation Accuracy")

print(scores.mean())


print("\nOOB Score")

print(model.oob_score_)

ROC SCORE 0.8136046511627907

Cross Validation Scores
[0.75324675 0.71428571 0.77922078 0.82352941 0.7254902 ]

Average Cross Validation Accuracy
0.7591545709192768

OOB Score
0.7320261437908496


In [13]:
importance=pd.DataFrame(
{
    "Feature":X.columns,
    "importance":model.feature_importances_

})
importance=importance.sort_values(
    by="importance",
    ascending=True

)

print("Importance",importance)

Importance                     Feature  importance
3             SkinThickness    0.066419
2             BloodPressure    0.077542
6  DiabetesPedigreeFunction    0.088810
0               Pregnancies    0.094652
4                   Insulin    0.098398
7                       Age    0.110647
5                       BMI    0.197081
1                   Glucose    0.266452


In [14]:
parameters = {

    "n_estimators":[50,100],

    "max_depth":[5,8,10],

    "criterion":["gini","entropy"]

}

grid = GridSearchCV(

    estimator=RandomForestClassifier(random_state=42),

    param_grid=parameters,

    cv=3,

    scoring="accuracy"

)

grid.fit(X_train,y_train)

print("\nBest Parameters")

print(grid.best_params_)

print("\nBest Accuracy")

print(grid.best_score_)


Best Parameters
{'criterion': 'gini', 'max_depth': 10, 'n_estimators': 50}

Best Accuracy
0.758169934640523


In [15]:
patient=[[2,120,70,20,85,25.5,0.50,30]]
prediction=model.predict(patient)
probability=model.predict_proba(patient)

if prediction[0]==1:
    print("diabetes")
else:
    print("NO diabetes")

print("probability",probability)



NO diabetes
probability [[0.90157551 0.09842449]]


c:\Users\Shah\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\Shah\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [17]:
joblib.dump(
model,
 "random_forest_model.pkl"

)

print("\nModel Saved Successfully")



Model Saved Successfully


In [18]:

loaded_model = joblib.load(

    "random_forest_model.pkl"

)

result = loaded_model.predict(patient)

print("\nPrediction From Saved Model")

if result[0]==1:
    print("Diabetes Detected")
else:
    print("No Diabetes")


Prediction From Saved Model
No Diabetes


c:\Users\Shah\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
